# Capa Oro — TFM TUI (Colab)

**Objetivo:** tabla(s) finales que alimentan el dashboard.

Qué se hace aquí:
- **Cruce espacial** (point-in-polygon): cada POI / restaurante se asigna a su barrio
  según dónde cae realmente, no por texto (más robusto).
- **Accesibilidad por punto**: número de paradas de transporte en un radio de **400 m**
  (~5 min andando). El cálculo se hace en UTM (EPSG:25830) para que sean metros reales.
- **Agregado por barrio**: densidad de oferta, % terrazas, diversidad de POI y accesibilidad media.

Salidas (carpeta `Oro/`):
- `POI.parquet` / `Restaurantes.parquet` — tablas de puntos con barrio y `n_paradas_400m` (marcadores del mapa).
- `Barrios.parquet` — una fila por barrio con los KPIs (gráficos y tablas del dashboard).
- `Barrios.geojson` — lo mismo pero con la geometría de los polígonos, en lat/lon (coropleto del mapa).

## 1. Setup — montaje, rutas y funciones base

In [1]:
!pip install -q geopandas

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import geopandas as gpd

BASE  = "/content/drive/MyDrive/Master/TFM TUI 3"
PLATA = f"{BASE}/Plata"
RAW   = f"{BASE}/Raw"
ORO   = f"{BASE}/Oro"
os.makedirs(ORO, exist_ok=True)

UTM   = 25830   # ETRS89 30N -> distancias en metros
RADIO = 400     # metros para contar paradas cercanas


def a_geo(df):
    """DataFrame con lat/lon -> GeoDataFrame en UTM (metros)."""
    g = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326)
    return g.to_crs(UTM)


def guardar(df, tabla):
    """Guarda como Parquet plano (sin geometria) en la carpeta Oro."""
    if "geometry" in df.columns:
        df = df.drop(columns="geometry")
    df.to_parquet(f"{ORO}/{tabla}.parquet", index=False)
    print(f"OK  {tabla}  ->  {len(df)} filas, {df.shape[1]} columnas")

Mounted at /content/drive


## 2. Cargar Plata + polígonos de barrios

Los barrios vienen en UTM (EPSG:25830). Nos quedamos con el nombre del barrio,
el distrito y el área (ya calculada en el shapefile, en m²).

In [2]:
poi  = a_geo(pd.read_parquet(f"{PLATA}/POI.parquet"))
rest = a_geo(pd.read_parquet(f"{PLATA}/Restaurantes.parquet"))
par  = a_geo(pd.read_parquet(f"{PLATA}/Paradas.parquet"))

barrios = gpd.read_file(f"{RAW}/CM-Poligono Barrios").to_crs(UTM)
barrios = (barrios[["NOMBRE", "NOMDIS", "Area", "geometry"]]
           .rename(columns={"NOMBRE": "barrio", "NOMDIS": "distrito", "Area": "area_m2"}))

print(f"Barrios: {len(barrios)} | POI: {len(poi)} | Rest: {len(rest)} | Paradas: {len(par)}")

Barrios: 131 | POI: 1393 | Rest: 125414 | Paradas: 9553


## 3. Enriquecer puntos — barrio (cruce espacial) + paradas en 400 m

El barrio de cada punto se decide por **dónde cae** dentro de los polígonos
(`within`), no por el campo de texto de Plata. Se descartan los `barrio`/`distrito`
originales para que la única fuente sea el polígono.

In [3]:
def contar_paradas(puntos, paradas, radio=RADIO):
    """Cuenta paradas dentro de <radio> metros de cada punto (usa indice espacial)."""
    buff = puntos[["geometry"]].copy()
    buff["geometry"] = buff.geometry.buffer(radio)
    j = gpd.sjoin(buff, paradas[["geometry"]], predicate="contains", how="inner")
    conteo = j.groupby(j.index).size()
    return conteo.reindex(puntos.index, fill_value=0)


def enriquecer(puntos, barrios, paradas):
    puntos = puntos.drop(columns=[c for c in ["barrio", "distrito"] if c in puntos.columns])
    p = gpd.sjoin(puntos, barrios[["barrio", "distrito", "geometry"]],
                  predicate="within", how="left").drop(columns="index_right")
    p["n_paradas_400m"] = contar_paradas(puntos, paradas)
    return p


poi_oro  = enriquecer(poi,  barrios, par)
rest_oro = enriquecer(rest, barrios, par)

print("POI sin barrio asignado: ", poi_oro["barrio"].isna().sum())
print("Rest sin barrio asignado:", rest_oro["barrio"].isna().sum())
display(rest_oro[["nombre", "barrio", "distrito", "tiene_terraza", "n_paradas_400m"]].head())

POI sin barrio asignado:  0
Rest sin barrio asignado: 3


,nombre,barrio,distrito,tiene_terraza,n_paradas_400m
0,VITACA,Justicia,Centro,True,13
1,ZAATAR & CO,Universidad,Centro,True,5
2,HOTEL MEDIODIA,Embajadores,Centro,False,17
3,HOTEL MEDIODIA,Embajadores,Centro,False,17
4,MUNE,Justicia,Centro,False,14


## 4. Agregar por barrio

Una fila por barrio (los 131) con los KPIs. `accesibilidad_media` se calcula sobre
los restaurantes (la oferta más densa y representativa de la caminabilidad).

In [4]:
poi_agg = (poi_oro.groupby("barrio")
           .agg(n_poi=("nombre", "size"),
                diversidad_poi=("categoria", "nunique"))
           .reset_index())

rest_agg = (rest_oro.groupby("barrio")
            .agg(n_restaurantes=("nombre", "size"),
                 pct_terrazas=("tiene_terraza", "mean"),
                 accesibilidad_media=("n_paradas_400m", "mean"))
            .reset_index())
rest_agg["pct_terrazas"]        = (rest_agg["pct_terrazas"] * 100).round(1)
rest_agg["accesibilidad_media"] = rest_agg["accesibilidad_media"].round(1)

# Tabla final: los 131 barrios + KPIs (left join para no perder barrios sin oferta)
barrios_oro = barrios.drop(columns="geometry").copy()
barrios_oro["area_km2"] = (barrios_oro["area_m2"] / 1e6).round(3)
barrios_oro = (barrios_oro.drop(columns="area_m2")
               .merge(poi_agg,  on="barrio", how="left")
               .merge(rest_agg, on="barrio", how="left"))

for c in ["n_poi", "diversidad_poi", "n_restaurantes"]:
    barrios_oro[c] = barrios_oro[c].fillna(0).astype(int)

barrios_oro["densidad_oferta"] = ((barrios_oro["n_poi"] + barrios_oro["n_restaurantes"])
                                  / barrios_oro["area_km2"]).round(1)

display(barrios_oro.sort_values("densidad_oferta", ascending=False).head())

,barrio,distrito,area_km2,n_poi,diversidad_poi,n_restaurantes,pct_terrazas,accesibilidad_media,densidad_oferta
5,Sol,Centro,0.445,42,3,1542,14.7,19.9,3559.6
37,Gaztambide,Chamberí,0.506,6,3,1558,6.7,11.8,3090.9
39,Trafalgar,Chamberí,0.612,3,3,1884,8.9,12.5,3083.3
19,Recoletos,Salamanca,0.873,13,4,2420,5.2,9.0,2786.9
20,Goya,Salamanca,0.770,7,5,2099,8.5,12.3,2735.1


## 5. Guardar salidas

Puntos y KPIs en Parquet; barrios también en GeoJSON (con geometría, en lat/lon)
para el coropleto del mapa.

In [5]:
guardar(poi_oro,     "POI")
guardar(rest_oro,    "Restaurantes")
guardar(barrios_oro, "Barrios")

# GeoJSON con geometria (lat/lon) + KPIs -> para pintar los barrios en el mapa
barrios_geo = (barrios[["barrio", "geometry"]].to_crs(4326)
               .merge(barrios_oro, on="barrio", how="left"))
barrios_geo.to_file(f"{ORO}/Barrios.geojson", driver="GeoJSON")
print("OK  Barrios.geojson  (con geometria, lat/lon)")

OK  POI  ->  1393 filas, 7 columnas
OK  Restaurantes  ->  125414 filas, 11 columnas
OK  Barrios  ->  131 filas, 9 columnas
OK  Barrios.geojson  (con geometria, lat/lon)


## 6. Comprobación rápida

In [6]:
b = pd.read_parquet(f"{ORO}/Barrios.parquet")
r = pd.read_parquet(f"{ORO}/Restaurantes.parquet")

print("Top 5 barrios por densidad de oferta:")
print(b.nlargest(5, "densidad_oferta")[
    ["barrio", "distrito", "n_restaurantes", "densidad_oferta", "pct_terrazas", "accesibilidad_media"]
].to_string(index=False))

print("\nBarrios sin restaurantes:", int((b["n_restaurantes"] == 0).sum()))
print("\nDistribucion de n_paradas_400m (restaurantes):")
print(r["n_paradas_400m"].describe().round(1).to_string())

Top 5 barrios por densidad de oferta:
    barrio  distrito  n_restaurantes  densidad_oferta  pct_terrazas  accesibilidad_media
       Sol    Centro            1542           3559.6          14.7                 19.9
Gaztambide  Chamberí            1558           3090.9           6.7                 11.8
 Trafalgar  Chamberí            1884           3083.3           8.9                 12.5
 Recoletos Salamanca            2420           2786.9           5.2                  9.0
      Goya Salamanca            2099           2735.1           8.5                 12.3

Barrios sin restaurantes: 0

Distribucion de n_paradas_400m (restaurantes):
count    125414.0
mean          5.7
std           5.1
min           0.0
25%           2.0
50%           5.0
75%           8.0
max          36.0
